# Kaggriculture | Adaptive Replay Agent | Topology Repair

A controlled first upgrade to the public Hamburger replay. The base 720-step route remains fixed; an observation-driven repair activates only when a pasture build is visibly blocked by a weed.


## Method: Topology Repair

A replay policy assumes that its farm geometry evolves exactly as in the recorded episode. A failed `BUILD_PASTURE` on a `WEED` tile is not harmless: it spends a turn, prevents animal placement, and can shift every later movement and production action.

This notebook applies one minimal closed-loop correction:

1. Read the actor position and tile from the visible own farm.
2. If the taped action is `BUILD_PASTURE` and that tile is a weed, issue `DIG` instead.
3. On the next step, reissue `BUILD_PASTURE` only if the same actor remains on the same now-empty tile.
4. For a late-day farmer repair, preserve the preceding farmer action through the end of that day so the tape does not drift by one hour.

All other actions, market orders, crop timing, herd sizing, land purchases, and geometry remain the Hamburger replay. The intervention is disabled after step 715, where terminal behavior is intentionally left untouched.

```text
observation + replay action
           ↓
Is BUILD_PASTURE requested on WEED?
   ├── no  → exact replay action
   └── yes → DIG now → conditional BUILD_PASTURE next step
```


In [ ]:
from pathlib import Path
import ast
import hashlib
import io
import json
import py_compile
import tarfile

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
MAIN_PATH = ROOT / 'main.py'
ARCHIVE_PATH = ROOT / 'submission.tar.gz'
print({'working_directory': str(ROOT)})


## Agent

In [ ]:
AGENT_SOURCE = r'''"""Fixed public Kaito Fukami policy from episode 89454731, seat 0."""
import base64
import copy
import gzip
import json

TRACE_B64 = (
    'H4sIAAAAAAAC/+1dTW8cWXL8LzzzMN1NSpRvHKl3R1iOJOjDxHogDAbYNQwY68PYN8P/3dphs7q7MjIyMl9Wk5zhSQ1S7HrflRkvIvKn/z37919+'
    '/cfffz37l5/OPlx/+nT29fzsP375r7/997cffPv4j19+/c+//8+3zz+d/fD24/bbbw8+fP/lrz9fv3v74/XN2fnZ6/e3Z+eb+x9/2m7ffPvhj9ub'
    '9+/Ozl98/fp/54dP+v7L25s3P3973ucvv33X9Mifzm63nz7/9u3v3n/8/MPZ1+NG/PO7P3x8/+bL68/fvv72h+31t39X3/nN+vTDdvvh23+Zt+DD'
    '29d/+fLhvt2royZMv5v+eN+c8/sWHrfr0/bmZt+g9X0zTFPX+5bOG3Rz/Xq7aw8ekPmg9bXJG5v9nx60hzXj0/svu0/z8W1q2p/+ua6OmrN/uG2Y'
    'meSfzl5f/zb28aKyozLbHYePRp/u/n+t36v44dvr431yPl9CSgtW+sjfP+Zwn+yed7dNdi1493na9/vx9rZzeTzsOpgedHv9efsRr8umobhf5QdP'
    '3y/8abuCvbD/wqaNSpaknQqzSLt3ARp5adrDZ0yDCkb3bin09MUOGhtGYVUNbfH9eC6/pu3T6ZIeed+sxpbxwb46PO+axmHX12B989VXHojdGaZO'
    'wt0roNbxzfzZu/cJfDbYFndTprxjNiML4aD/0+Annp3oP3v2wUicqN/T27306Ey3SaxX7d25SRbCt3ch6t/Ez+GB7BQxvH5/c7N9/fnnP20/fn57'
    '8/bf7g9d2iZ9k9daN4/tnSgTHcQdUf5+cIJRYk0pdth50n4rTG2yJ2Cuk/hB0+Z78/H9B7gL4VI4+BaSb+6XBTxqpkdO2z39bCV0d4+V6KCxyf0q'
    'k6OlnhZ3o/jF9SyjpycggQhfsT09Ak92X6ypLrG4aeR7wauipb023JmO2UUGnkX3I/0guVLz1z439/E3txjMXfz2nkFvsdOEchd/lFCOBnXLh3J4'
    'cOYQHR6m4R5HTwd4Rkcwt28AQkyzO2oVxehT/CYBwTRkBPEOQoSm42H+o4bADcAP+8cd/yQXuEnYQtyhSghHIQW9UyCScueef7F/9JZjOncIg1EN'
    'gzo3K0mNnZ0U+r16g21Ux76YZ1bVuK76RP2Lp4PsiXyvzWlnT8qntORp9gN92sG5+Onzx+vb77cfP/51wSee6GtHIsHniO8ZvOsG7w7+w4nAO+1O'
    'vhjvTUcamN9GmI5FKNlwb53YyOBpI9EeSJp3Ud/BkbtcxAe6tY+jRwI++8V+cNmE1jkDZwYW3pfm4z0aPya+mYXhYPASOxaEfH5G0Ts9EowznUcj'
    'MQEYrOnrEuFx7ouHgj8bS3ZepgiP64lhyfeOzCf52gXRPpNfPsd+Wux3giBwkJW1TDjYigRmA8PlIEEhRFwkVkw+7iRR42BWLL3XvSCyLZqs9K4S'
    'SDrBF2KIFbpK72Y5wXSZi2EWNisDHUaXb97+WY0Ai7BiEC2PdQjcIZP4r3W6lIvJfWO8cGP0ijLJziv0IcnBG+tEzyMyIXX/E1qm2n5JGLAuFaeu'
    'H1eceu6+GlfPMewfNoadnnT6ENbFHxeJYMMA6FThbO69rW/aEUy0Gs6CHmjo6MPBpMPh7PQ1KHZ+gHA23z0NK00lJqeeISl+TSswxqUev9PQrOcJ'
    'JDZb/AELBZfT60X5/gUQ4kq3fkdMyz9uTCvQMU8T02aJmaelaP5w/fFfFbpqa+dZhJt+tKe2QRf6SRFzmsLpR4/oVbv7VAsT+OtcT49y3QA3kwkv'
    'j/yxF1wyghHcHzbv398AexL93p3IAeY4rdR56KqyzhzRNOxGixvtLzRkzvsko582a0ZSlicUQftP1gzE/mTWjFSkjDxPpu8DOyCzCKhVzTq+ILC4'
    'rwmkq2xteyr7wXoqefKbip8Rk8dcPx2wWsKYt0htIZGz/gDmCgAepTxhII0Ok4HTYcMbiw2/OFEc/WI0CHWtiZJx8sVycXIIsLKTPZOSVMHYFl2T'
    '9U7oSyYKrNv0S2EsswBUCPQn5Sgfhdo06CbGMil9faYZfgBeTjFACE7Bz6YYXMgpsvfnR45iKH60AW+pM/b0U3D/nozCRjW+edcqGa6yVyhAb0EL'
    'BpInQLkwxPV9I3YDw2+vtO1IFF6g067wqIs0nQCUx9YWUcwJd1ZS7CyJ55Ldq8jo2CMGpo88Oya0SImBothb6AFK8Dwki2GRzQsHe948uZj5yEz1'
    'AJx8LIHz/atsOmyHuApTy398e/OX3TD2RtIAAtKHFdntJoHw6cVz+NS5MUAU8eduPxKwFYiRo4klyJI8m8I7dRq3fXuuMbZlf0XbJIRr2hUPbAD9'
    'W+maZTdmN9fv3px9reAx+1dJIstNLH8fsL0sYSygCzDzlCOXy3gHKBCqbSmxg+V+ggnvrelhxPL5cCog7C0PlWCVyUyoDzKlwFdcSYLz5pKstSzf'
    'OWwkiLwkKGXAfTOiboPV4Cfdi/l00qTe+mMjuL53IO1sT8MDXqh0JQgGw5uRt5c+w2wglVdtfhg1nh4b2uDqRx3HdQK9oDIKclNU42FsGtTK1st5'
    'P8X+JUdtJDdfx9QBFjOVYJKOdrEZRitujqtWWPab5T2EX5JHQJqSmJtWgX+vNS8TrSldqcDElhLuhf68VIqQtFGpYBdgLAFmpyMGZ7NWy95hl8QL'
    'E3H+Fr6S4j2wuaKurahMyKpzHkBIsE3PjM53a7gAgxcJQNZS2B78P1S5HsL1rvMmQn9ZuH2S/PRYK8FmLfFfpqdOgAtj+OvMxg5+HDEypl569Hhy'
    'L8SadNT0fUe1IVAvEsll4o3A9CJg0sm9HRWD15k4BPwD0wwbwXhPLec5G0QbwqIjBKzAcHL1lblODKptL5t0NLYJrhpJ2pmfDXoW/zRAZVfyTuoR'
    'BW8O8eRmSAvkxpWNHaXQ+bYLzRah2uyyehTzNndZTWoUQ/A7QfBasrwkV6ePoxXWAprCAr8xzy/cK9mmFHtWGrOu/Kln2UrJTrldm8ebb1+lLpSX'
    'zbcPf2hxK6znqUzj1SI5tu4UVTYlWGwVFeipeDo6k9Bg2UxBTzFfBn0e0amtRi+a2d2zOD8dcIbnpjANdzTudFzj0CkvahHlQTHAAr6HXxwOx6DT'
    'WIKYnANw+zc5fCuEYRUujMvnht670UTNHuI9A4myG0LUdIhCrvIoN352VgMowJORnds0uZkxLHF5qVisALGs61dvMAigqBBo8kywT6hEG6Z37yxZ'
    'poCP57ygyV2vcGecOva0cq12G3s4D5acrRghTI07R4Nv5aSbWXy9+/vLrzVwGB1ch51CPGHj0ByWaMBsUol7yCrbAoIVbjvauCIpxqciCjV55nwi'
    'iq1RESU7elzWUvXugI4wbZNdG/yAVNQIZBCJKhigLaBb4Avsacpafrh9hQVBi8/vd9iRvHDjtB28xezBYwh14fhfDaMX952/sryE1SU+5xZnC+C3'
    '1eWT5AasNo8UrADr1udPrhvupPuqSHPwxQMB0hftdPa54KFz9ngLdSIB2H3yJdlF50yBVKNiH645RJaRHJZbuEOZEqAsDousg1tWewXHiKAopBZX'
    '3/gkoCDYvlP3aKUNGKPlIu+FpOIGiD4tF9OqfnGUjLj/W9XdQj2FOExDuc1UPNFCwlAHm8klWBRPQY9ZxDyuWbdbkZbrJMxdOt46OTVhvQnGEw2j'
    'legpzBgpfo+3Jmi/MvRUkUbz05KJmF20wN9oq6zZCIoaAL5oEmqJO+Zk87NOxZWtW+lMxdWSQQJBVgJhBzckg1fETIXHOkdYdHH5sgAW92gaoAMX'
    'Qmp9X6GCXPdrnhguTjxUCThj3akqrOzsIFM9Ug46fzHF+50yJr23Uwo80ohdn1phzyWkzeg6w9tMl4eAAy9vKe4z2HQ/3hekQ+s2bstFDcR4QNbL'
    'xTNatAhahHnESdyiChctgRsxe7gjk4QKfuTfgp+YIgMWps2aNU+KcWhFJclErA0EJBH2CYUCkhZaMlUmYoDh3ydd5SrSesk+1iJE0VSIpBDar1T2'
    'ZF/JvhBo7WkveFs1nVAlqYLDAAMqALukskVA3MgvonXGdVsD3M73kb9Nu2jrnXMQRF85Zhbhtu2jPhzjhSQt+CnhCq1EUrku1ek+ngisOwonjliH'
    '0pUjtMEeRxxbiBVOLGcpMv+Y4F0DvhE+QbOPjOVPrShpxOk4eLnv9oiyRRhlzo0pix7utP4neGUwpDnU3zDV5gAljbA93LSUc7hSR1jXwQWWDWrc'
    '8VJiTk4JZdRQV0iYIkg/8fKC4Eo4qQmDzcSZTFzl9aNUldJJMjzNwnyJntuaSj4tSRfGZe1tADWxDPHtesSNQQVvaQWijNSn2kRbCuyhNcUyxKm9'
    'qaArNAv1Si9r6Nl6KWeWS7elV08ST/NH/sXjg9PqPKCardTViRC2EzOzFuRyBDPExBt7LMvCUYxNNIa0iUSsCGjTuWVVA5W6XcMiVZU7usTg0u7l'
    'SFgNBMtjMs8qWJomQzmj50X3bKYQgWO+uSpQKe8TtSbcCpolHUzWWXbddKqojfw8PJ5CXtZG8+0sOkMR4yJNlifMcbUmdbQjKMNLJLCNFQ7hjD97'
    '4mDhm2Z7NVAZgni8cCecCLvS0Rzuo1kux0i9b8D9hygEwv1OOVrWMR5UU1lMBXG+jNJRySqoQ9ILETMCI1r6CCsBm5iVVMIPEERuO8fscTSHGz7H'
    'TfZsOmIG2jr1UeuQxrZ1JGhJMJE/H3n+CoY7NQBmYImBawy/wq3SgxB67KJjJg6E+8vNaN1N/0/oZ3bRHZ0L33VVIVdhyGMi6Pky3DJPw8kvqB6Y'
    'Wcaj52xN8gfhkiXMkRKwX2AUMjR9m2X7zj2gtMIseVpZOTWKnJVUDoc9y2G3/dSvtwcW67D4G7hSxI0PKFhtZLhw2FUlGMurEghfP1RisSiKAIAZ'
    'wslaixmWUyjL2kWDxlNII+QsWbblklhhDuOlki6x3H2QU3Z0jxxEKFjWRboeDplw/xJbblccVd/Q5FbjlFVy87AMG2MsggsOb1R1B5KB8kwMEkfg'
    'n75FgKH6rHIcuVR+ZS6VLzM5IeDiEi63xhGnY9AM0onwOTUqD7CtTvCHDDcdW8GgDJAE85Y8lrECMA4YWlChUqKuX0UPmZHySmVr6dhSklXF4ZiV'
    'B9fGlzkGJdANqaK7ab7FvmjzZY4hOBY5aEMcnecY2ogpuPX9psWkkVI2B6Bb9tDhDF3irq0rXUvCmFQEfD9Dd+2J1twxY8nr6bBEcb8YiH2Vacuz'
    'idXvVpbI0SMUjT5CWWKEFEGFHumb3vOHWYU2iqIED+YD2bnqWNIfqn2iOewrt11mRTHBiXhrjpytHoynYvlPAERSkT93TBpWm4qTMi+SQAK3f0sX'
    'SFAXX0eNuljtnnDrkOhXZhDWOgB8zoUiTlE2kcFh0mudOwRTPRM4Wk/M0BSPFKZ5q1BIBvhFWW8vbNaM1gZFyprqXbERpWLJUEAc1EiqI0OHwyaP'
    '6gHIsbtaD9/cnqp3tkdy5l6awfXWXcEm18JrgocYBcbD4PS4/F12iyAZJ7dWF6B4C1LqhhCwL8ndzyR3ptL6wvwbLK6dNAJn3lsdE8F8BRXPbLB2'
    'GFLVMd6sAEHBbhS97ECBs93RdetfDyZKrQPcl71WS7VgC2iXoSRVAgmCDd19P3jwhQMQuV7oz6DRYwONOIWqTXvnX988AsrR9EMjd8ZGVnkOEo/R'
    '+3NivfS7eMPm2/qME0SAii4w1GIhUEltliraSo2N9AJ04njH1hAxqWg/eJCgEzhSAQ8rO+y22tsBepLX91W8tpD00CRYvNw0iO26iF32nUbXkUCg'
    '0/AVNwAlw627amlyR92hgTLvyipQksSzQRSlQXxJJY8b0lSG4qvaNnbmDHALmHREpsASDS0adsf4L9yuc3W/PgGsZB8ttL7VTakW1FMpVmQsnUne'
    '0If2CUWXcXjBkBt+1XhpFo+7PdkoJCcowgbMjuCIkRhdOebE1At5pphtITKxAQb10h7n0vXaZF2MyhGp3T5DwWg0y+shjczMIayFuaJsfoBKbBu/'
    'wGtTY8gs998ra3jp3GQJPMLMXA4VB6XXFVMjw1nTaHCMaXQ4M9DsSZ6xBhsok4UDhOlFnw7uKQneBovRPzX4STZ0etxEJR1FO1xqAirl/xfFySV0'
    'fAoi5Yq4b9jAKsnEJz5QPboZTqADN0rBXayFLiK4sQ09HDexilJ/sbr98nZPooOOZGSTn58e3gfB5wCxBSAYYYoeTvjg0iMXlazao4YWhSF2B2uL'
    'k+Twtb2Yv8pRh64LcpxRZJ0WCr2dDK1FsajozbTFQFMWrHCKR5rcUjom3bQCKDe0T3jcV4AWHbWV+YQakZHX2qriXwL8HaVLcJu68FjOB4wFKuqG'
    'cpewm6dXVspudWaALPFehZk6zVVOVLcyfneR9vvSAPj4BGHKQtFNnuqZ6QKPV66d/PTgeecf9cTKUPGgR04GeRFd3yhXyWI2CXU7WwTMnS24PkOl'
    'OTxSWqh4fKXDXFWO0SFnyLqRRx5Iz+yiRwnvcCVZV6E8951yWpgHgjcjPKPg4pRdpjQK1CjkmDQ/EPkVjQbefAUS6hEtmEKLeLcaPkfUENlkW8My'
    'BiGMqvNzIOtCKscxR4XRXcEEc4LgFOJPjFTje2V1CxotrnWrGnnxOGSYmNcDAoLwDUKyouN+GC72rTkK5tnZkszpSfW6JVWmCV+kIXQKnM7TZPcr'
    '15h+RKyjgDIkZmg86kY1ZRNxNTnXvsvO6vF/dUbc3v/WXagOH4es8SgQQXWEJdEgY5RQCazKbSLkmbpARysfxx4dKAq9tcQWdYfZe8RNdezbtX3q'
    'l/Pr0XWJdDBFMs3tmcYgZUmQhtpB1xF1UODZHjTaD7VeGl0SOOTLvHLCwiaF0DgiInRMoXFG/Cn+msoXPrhqwo48GdqRbs2tcPeMHf3O7Yw4ePRE'
    '7IwE7EjHi1Rz7MH7/cEleUs08ZTV1IUdpa2NQm/XPGxzYlJJlOuKFlT0lcEgrSWRDYLxQfw1ASwhI6qFkBrn/jVaetHv4Q2ynqafBrsRaVua+RNN'
    'xPrwQUUqFCIaQYaoW+kDMgJjjWmcFF3rVVbhSiXJospPkvHRMRgIII/RvJ/KkRhyxc2NIjiIlqOLkJsoAwtM33hmrGRqpB48o2Dg4xyedcBZd/Y7'
    'Al+t1gLnwIGpaOYeEJMoK6GAEAbwgOQATX8JJiS1nzS7IdQNvBRYDay4fWCSwcUQGBmOawDChlbU05HQVDy2rMENGCHkG0f8nO9+J1gdJ7ClaZSp'
    '3zR1g0N/eax+S9kFMQ2dVPcuK2gLV8CLYZzlCFVZS6jK1cNqri6fNVc1zRUnTPwubH5gEu8DLJBvwe5BipPfoTxS6m4Ft/jsciEwBDpJEsgXHosG'
    'mdwK/KgNB+PUo8jd5XBKOSk9LlFlYI6TMElKblTutRpTBy1AF6EACRNa8cvCCAFsJZGFACXnxFA8IK6R1D8pjOzChz0yFHNMZdqwvFXGl1cGYwML'
    '9IoFlCwRYnPBYONwKXoEwCmJ7uE4hC3yIRHbPVaCy7PhiHUYBPwQq3lNNqiStJVSwcYAxaACJ8Ua/CSuhzAC37hBZXpevzE5tAMeLyK0EGBgsYKn'
    'RKMroXkYrwkSaIao5qAnVm6d3Y/UFWQtSwPCjrtlDVgrSXN1VrY4dJxarV0X4QQRDeAh+34Rh/2cUFHQmJ0PMNNI3T2LnxUcseneCeGil020nPUR'
    '/WZXX96CR16p+6P/urTHzxiL5+KZxbO0v7RqFPSorX4MtNT2m5Uu41m8pJnmR01XL0xofPbPIh7UMkxGmS1qyeyUQFi3XqA0HQUsDGUlFVXYrW9G'
    'bekSUcvA2qLylv7aauB1orbZJnKOdYidCKJNr8Tk0CNcRxRh3A5iW65wKJRGEW4L77PfKJJC2i/Tv4T6KAG3kCCRp0b3fRO9mgLZcJ1WCLwifBWj'
    'WLIsVhyBdVVzHl2lbUTU8NzZz9y3Qwrpm3qaJDFQ5za4+SMW0ZBlLMAW9sPMARLcLuoOxHVLoZ236oy9TvngsjkKGVwU3moy+l4XrXBDlzHq3BWd'
    'LtKJsdY9vzWXGUcZl6xE7CM4rDOboRmKrXxsL47/hgFQ+++bG9xgSAnACVYMFE7iReowhKRe73IAQDrove3WCKv5ga9rgkJqNsXcjdCSTHQkMTPg'
    'CHTjA4mnqNm4Z2cF8tnOHZLrecd1KN0f1lXplQK3eRbc96/jFd52zkyeApiTi06sniZWx4pPPC2sDh0eCZLY6ilhdVBXdOr6cLo1EA3yb7e5NK5N'
    'fKcUinPKw0W5+imLxd0ShobmYa7IH/vLw8lEtVCIaslPcmXp4XkgVaZoRosuagXNXE+FOIzU2itNgrfF8tj86EbYSEzyIRIom80GfKfRut2C7bRt'
    'isWreXZNM54OohmvKk5Wv4YjhAy5ZNm11YjfTwA8EbFMqLJiBJkmgEpZXhIVjb22YbzBMLCmEm5cyUEksHgCRViOHr5LTJTH5fMyUlGuTK2Uwhny'
    'Us+eknWqwU2I4wOkZz6uGv2mDG5JE4s+mYskj5uPUB2JY7QALhId3czoCB+HENAaKmOXgq2ogdH9FEXCT2551Apc+df33AWLNRq9sUodGZoTpd5c'
    'aENPANP0jLSgbxdNNe3uNIioSRdOm14CqI2daw+qsNw8KywXqWqn/vHDmp1317Rb3N/qNAXumPUM5ydtVaudQrbY0TMmdRtwwlraP11mF+ZdPiH4'
    'OADDiWdKsUyfPEccla5iRiIPUxNQUe6l1TrxQ6CaM9gBRXa5rDxZtEkcBuCQeFfW9YE6X772PWZqsd3XLCGygpfIjpilQKUqiUQVcWmuTjcpfRcI'
    'oTPlU8PyhD5JvGeabhWXuYQCL1iL4S0UYnyYo1YiT2rGXqCTPftY19hGMKKMxXW4jou3HogvKbqPgYYPeWRtB/ymoqEl1mQk09yYM2UI5oxodeHR'
    'Qa74XUCKeBk5xyRAvqgMNpolIJJmb4jQGopiWCwuEJETSgccap1HeKUjw5xXXE+sVAPD0pP8RkfS794K61GBl172em69orLJuw1y9ext/qyKbGRa'
    'bZ4804qi0kvY/nZTrYLKJP1G5wNcK+KktRSpJxzusObatmxsFl5bVqWQyK/MJCT8Vl0qBhauGfI+KHjIyeXkMiyfouPcJqv2IlykfZTOlXiyx1YB'
    'rZJqZM1TWbuxqaMc1T237G+p/pQVZLJWJy7qB8zKpEprvq94lJd49AkKbS9GvrILSa/cp6UupHpEU6/sySaW7uN7WCZQDM7SOkORY+gHrYLubRn3'
    'Dr9JRwfQT3ZbJDUtSvpybKvNkCgQolWWChVZWAUGXpIuEGSS6R4SeovCggpuQxkPe0jxKPQDTtTUE45s5awB1Z6MMfx2awyLEzTPeQJ6g/oJeVut'
    'SwBhNHAbQTk80xFAOuJuu3ZJuPVVy3zGVLdXQhlAL/KgYshbv1jG9Hdz1a6C6HqCyUvJYD5L+LqrBjh+FR+DdZdUlGwPXUNLUw+x1foBtJV0HJ8i'
    '4rc5hVj04XzQ4uDs5QkmtA0C1EvnUcBnKN9+WZx7DfQDgjpJs9buhSYS0Ir+zo0VBIRqxInVIjHddDrPDLHosbQiFAVN3ledKwI51zkhFtTRfIf4'
    'zd8S9CkolY2EsSDNpvk5d5WOYNGeBWZX0Qjeo/ykvj0IF0x3wryVCGJxLu4a1AdkY2ugxmCaYsE+RV2yKH5OcFzUSqvmAXOSKJErB35fE3bgtwpH'
    'ieEAAX9JXFErXdDieKTZiyS8njCCZaaKFjYckotJHnZbBebLeOIDdCeBgPrTc5ng9qUrL95qHvqJjlzq0wIYWgAn4Dy+4P9J26lpXsAqc6VqwnKj'
    'K9RH35rmhnXGI1ohz4Ajg7fdGcBNXiV0zMd9srBPz/weKEytix27W6ARhgaWdew7gFuDtZu8JEkz54pz+FKfwyHkbs1xuqWLEAxLMi+eJZkPKsl8'
    '1MUHFtFk6m4wT0mWqfvrj9jiP4Aqk7q115x9qsUh8cgKnC+RAANDjw63tHTxC/CKJFMDt9bDVItkAFYVUwSBhTrP/bVKCa1PU20HkVBJ6phccEyj'
    'hteSLvsNUuNCEUk74lM4nWhXkChSuRW5j75q4cUxa7gUMqpRXsKokpGiaGVabjoo6lB7+JRIWe2QKQkwoBtIhNUWhku/RZLAW8Uknl8UdNKH2Vq+'
    'lYxt6dWlAQ6aW60Qa2N/e0BxUmuSzs4Z5VaJASd+jRlIaQJvrmMrM6a6AKUnFFDbgvikXZKiWC8iBYX/FB0dbZsDZ+j3uPa5saeCZWG1NJqRrzRk'
    'bZWxDoNucnRBTX9h2WcuvKwWJhWZZK18sSuliMWzlPNxAkv7k2OM2BVAUGKhxuVBJggdHYGDeZEY5YIvotOk8Ga+tlcYYwoMrgZQhUNEqLydrFWj'
    'SrC6bbU4TQloMaUJDVTDnETZAerJPmkM6CN4AwzEFoL0GG7EM05q4e7MSJ0uJK45wr6kviBiva6l2i9APeERHuXWRIR5u13CxQ8dQ5FtGUulw23H'
    '7jpJL6hzl6Blytg1E3VKUWRLitGqdAxWDK7QNtAQIrEiXCRKIBtCHRSHs6gOIAFQBkqBam7HAowyJVVRLVPuq5SsPykKhw5y1CpbFHAjNGSDWqIN'
    'XgtnZLHMuYpWawJ/6GKLJR4+IaK4JlOJt5da6sNMXs4QkBc+3ApGBCLFJsK8ym3TzLnAfdhsYaTwuFntRNU2jGhLc7AWVxmeCN1ZfydxigbRncYr'
    '1MsTcW6eBLozCOrAH2KfTS8GqKI7492MguREzq27mzNPpfEu2XjSXp+rkZnuISrukqLfkZJAwz71uVqti+cCDXMtQwD3h3F/qOvFMupItRo7xUZg'
    'RlIb42iWhSiLO8KTUP8UbBeKGDPTDe9DKkElvBi1QmNm/3faLMvrt3LgB9gFSCkySArAHFXtiM19Rod6kL5iG5SxYKqo94S1Qes0SUoWin2qdjD5'
    '2k0SUeuY9eFj1YwnMGXOW+ct1YQl0YpfhlkhIot5SboiH+IKM7UcmKuN6R7aSOSkAbfgd7y5cDknTEEPwLiIH6LXU5jLnCJJVpn1Yegu1GWKFcqz'
    'jJ04Drc2UMD2yidkOisGrAppPLWTLuwDdVljZfbQEW0pRWpJznESzfyojokzLwy04lXWA8ZiG6qW8vJoSNXjATKKJLKPBeGQ8lARmaZtUwaA0JkD'
    'Vhel15cu3LygCYekklFfpkUs+lMrNlOqop6uwKmKo8ugOHN41ZfDr/dy2q1mjGBvoe3ZTQ0Vd6hw3DTlAsJ38rTvOgDVx+6cglElu78K6/ay1ikw'
    'hHWVBM1x3QGxINklLpMJBS8DwQfSPgkMw3BTR5phf/I4WmF/Mn3gKjuFKE3aRkjPUtvSZjHHzpR6Q8nkKA2d/Z/WMSQEavIT76/Cpr0a2pm9H2ob'
    '4rkNz234I7Xh6/8DOiOf8m3fAQA='
)

TRACE_ACTIONS = json.loads(gzip.decompress(base64.b64decode(TRACE_B64)).decode('utf-8'))
REPAIR_ENABLED = True


class StateMonitor:
    def __init__(self):
        self.last_step = 0

    def observe(self, observation):
        self.last_step = int(observation.get("step", 0) or 0)
        return self.last_step


class ReplayController:
    def __init__(self, actions):
        self.actions = actions

    def action_for(self, step):
        bounded_step = min(max(step, 0), len(self.actions) - 1)
        return copy.deepcopy(self.actions[bounded_step])


class TopologyRepair:
    """Repair one blocked pasture build without changing the replay route."""

    def __init__(self):
        self.pending = None
        self.farmer_shift_end = None

    def reset(self):
        self.pending = None
        self.farmer_shift_end = None

    def tile_at(self, farm, position):
        if not isinstance(position, (list, tuple)) or len(position) != 2:
            return "OUT_OF_BOUNDS"
        column, row = map(int, position)
        tiles = farm.get("tiles", []) or []
        if not (0 <= row < len(tiles) and 0 <= column < len(tiles[row])):
            return "OUT_OF_BOUNDS"
        return tiles[row][column]

    def apply(self, action, observation, step, trace_actions):
        if not REPAIR_ENABLED or step >= 716:
            return action
        if step == 0:
            self.reset()

        player = int(observation.get("player", 0) or 0)
        farms = observation.get("farms", []) or []
        farm = farms[player] if 0 <= player < len(farms) else None
        if not isinstance(farm, dict):
            return action

        hands = farm.get("hands", []) or []
        hand_actions = list(action.get("hands", []) or [])

        if self.farmer_shift_end is not None:
            if step <= self.farmer_shift_end:
                previous = trace_actions[max(0, step - 1)] or {}
                action["farmer"] = copy.deepcopy(previous.get("farmer") or [])
            else:
                self.farmer_shift_end = None

        if self.pending is not None:
            channel, actor, position, expected_step = self.pending
            if step == expected_step:
                if channel == "farmer":
                    current = farm.get("farmer")
                    if list(current or []) == position and self.tile_at(farm, current) is None:
                        action["farmer"] = ["BUILD_PASTURE"]
                elif 0 <= actor < len(hands) and actor < len(hand_actions):
                    if list(hands[actor] or []) == position and self.tile_at(farm, hands[actor]) is None:
                        hand_actions[actor] = ["BUILD_PASTURE"]
            self.pending = None

        farmer_position = farm.get("farmer")
        farmer_tile = self.tile_at(farm, farmer_position)
        if (
            action.get("farmer") == ["BUILD_PASTURE"]
            and isinstance(farmer_tile, dict)
            and farmer_tile.get("kind") == "WEED"
        ):
            action["farmer"] = ["DIG"]
            if step % 24 >= 20:
                self.farmer_shift_end = (step // 24 + 1) * 24 - 1
            self.pending = ("farmer", None, list(farmer_position), step + 1)

        for actor, requested in enumerate(hand_actions[: len(hands)]):
            if self.pending is not None:
                break
            if requested != ["BUILD_PASTURE"]:
                continue
            tile = self.tile_at(farm, hands[actor])
            if isinstance(tile, dict) and tile.get("kind") == "WEED":
                hand_actions[actor] = ["DIG"]
                self.pending = ("hands", actor, list(hands[actor]), step + 1)
                break

        action["hands"] = hand_actions
        return action


class ReplayAgent:
    def __init__(self, actions):
        self.monitor = StateMonitor()
        self.controller = ReplayController(actions)
        self.topology_repair = TopologyRepair()

    def act(self, observation):
        step = self.monitor.observe(observation)
        action = self.controller.action_for(step)
        return self.topology_repair.apply(
            action,
            observation,
            step,
            self.controller.actions,
        )


POLICY = ReplayAgent(TRACE_ACTIONS)


def agent(obs, config=None):
    return POLICY.act(obs)
'''

EXPECTED_TRACE_SHA256 = '557755a8d0f06e1203c231c7b7d2ce4b572a1540c4a73999dceeaf9fa347fd23'

agent_tree = ast.parse(AGENT_SOURCE)
assert any(isinstance(node, ast.FunctionDef) and node.name == 'agent' for node in agent_tree.body)
agent_namespace = {}
exec(AGENT_SOURCE, agent_namespace)
TRACE_ACTIONS = agent_namespace['TRACE_ACTIONS']
assert len(TRACE_ACTIONS) == 720
assert TRACE_ACTIONS[0]['market'] == [['HIRE'], ['HIRE'], ['BUY_ANIMAL', 'COW', 3], ['BUY_SEED', 'MELON', 6]]
assert hashlib.sha256(json.dumps(TRACE_ACTIONS, separators=(',', ':'), ensure_ascii=True).encode('utf-8')).hexdigest() == EXPECTED_TRACE_SHA256
first_action = agent_namespace['agent']({'step': 0})
assert first_action == TRACE_ACTIONS[0]
assert first_action is not TRACE_ACTIONS[0]
repair = agent_namespace["TopologyRepair"]()
blocked_action = {"farmer": ["BUILD_PASTURE"], "hands": [], "market": []}
blocked_observation = {
    "step": 12,
    "player": 0,
    "farms": [{"farmer": [0, 0], "hands": [], "tiles": [[{"kind": "WEED"}]]}],
}
repaired_action = repair.apply(blocked_action, blocked_observation, 12, TRACE_ACTIONS)
assert repaired_action["farmer"] == ["DIG"]
next_observation = {
    "step": 13,
    "player": 0,
    "farms": [{"farmer": [0, 0], "hands": [], "tiles": [[None]]}],
}
next_action = repair.apply({"farmer": ["PASS"], "hands": [], "market": []}, next_observation, 13, TRACE_ACTIONS)
assert next_action["farmer"] == ["BUILD_PASTURE"]
ANCHOR_SOURCE = AGENT_SOURCE.replace("REPAIR_ENABLED = True", "REPAIR_ENABLED = False", 1)
assert ANCHOR_SOURCE != AGENT_SOURCE

MAIN_PATH.write_text(AGENT_SOURCE, encoding='utf-8')
py_compile.compile(str(MAIN_PATH), doraise=True)
print({'trace_actions': len(TRACE_ACTIONS), 'trace_sha256': EXPECTED_TRACE_SHA256, 'main_path': str(MAIN_PATH)})


## Replay Profile

In [ ]:
import matplotlib.pyplot as plt

activity = [
    sum(
        action != ['PASS']
        for name, group in step.items()
        for action in ([group] if name == 'farmer' else group)
    )
    for step in TRACE_ACTIONS
]
plt.figure(figsize=(12, 3.5))
plt.plot(activity, color='#f97316', linewidth=1.2)
plt.title('Replay activity by step')
plt.xlabel('Step')
plt.ylabel('Non-PASS actions')
plt.grid(alpha=0.25)
plt.show()


## Paired Evaluation

The candidate and anchor differ by one flag: `REPAIR_ENABLED`. Each seed is played twice with seats swapped, so a result cannot come solely from starting position. The candidate is packaged only when it has a positive direct paired margin; ties or a loss fall back to the unchanged Hamburger anchor.

This is a conservative gate. It does not claim that a single short local league predicts the public leaderboard; it merely prevents an unverified repair from replacing a known strong route.


In [ ]:
import statistics

CANDIDATE_PATH = ROOT / "topology_repair.py"
ANCHOR_PATH = ROOT / "hamburger_anchor.py"
CANDIDATE_PATH.write_text(AGENT_SOURCE, encoding="utf-8")
ANCHOR_PATH.write_text(ANCHOR_SOURCE, encoding="utf-8")


def final_score(environment, seat):
    final = environment.steps[-1][seat]
    reward = float(getattr(final, "reward", 0) or 0)
    if reward:
        return reward
    return float(environment.steps[-1][seat].observation.farms[seat].money)


def paired_match(seed):
    rows = []
    for agents, seat in (([CANDIDATE_PATH, ANCHOR_PATH], 0), ([ANCHOR_PATH, CANDIDATE_PATH], 1)):
        environment = make("kaggriculture", configuration={"episodeSteps": 720, "seed": seed}, debug=False)
        environment.run([str(agent) for agent in agents])
        rows.append({
            "seed": seed,
            "seat": seat,
            "candidate": final_score(environment, seat),
            "anchor": final_score(environment, 1 - seat),
            "status": str(environment.steps[-1][seat].status),
        })
    return rows


SELECTED_SOURCE = ANCHOR_SOURCE
SELECTED_NAME = "hamburger-anchor"

try:
    from kaggle_environments import make

    rows = [row for seed in (9601, 9613, 9623) for row in paired_match(seed)]
    margins = [row["candidate"] - row["anchor"] for row in rows]
    summary = {
        "games": len(rows),
        "wins": sum(margin > 0 for margin in margins),
        "losses": sum(margin < 0 for margin in margins),
        "mean_margin": round(statistics.mean(margins), 2),
        "minimum_margin": min(margins),
    }
    promoted = summary["wins"] > summary["losses"] and summary["mean_margin"] > 0
    if promoted:
        SELECTED_SOURCE = AGENT_SOURCE
        SELECTED_NAME = "topology-repair"
    print({"direct": summary, "promoted": promoted, "selected": SELECTED_NAME})
except ModuleNotFoundError:
    print({"paired_evaluation": "SKIPPED", "selected": SELECTED_NAME, "reason": "kaggle-environments is unavailable"})


## Package & Submit

In [ ]:
with tarfile.open(ARCHIVE_PATH, 'w:gz') as archive:
    payload = SELECTED_SOURCE.encode('utf-8')
    info = tarfile.TarInfo('main.py')
    info.size = len(payload)
    archive.addfile(info, io.BytesIO(payload))

with tarfile.open(ARCHIVE_PATH, 'r:gz') as archive:
    archived = archive.extractfile('main.py').read().decode('utf-8')

assert archived == SELECTED_SOURCE
print({'submission': str(ARCHIVE_PATH), 'selected': SELECTED_NAME, 'bytes': ARCHIVE_PATH.stat().st_size})
